# 2 · Specialist Agents

Three **tool-using agents**, each an expert in one area. Later (notebook 3) a supervisor coordinates them.

- **Knowledge agent** — RAG over the help-center PGVector collection.
- **Order-data agent** — SQL over the Postgres orders tables.
- **Web/research agent** — shipment/carrier lookup (a simple tool here; swap in the wk4.2 MCP web tool).

In [1]:
%run aurora_common.py

C:\Users\akhawaja\git\cs4603\wk4_capstone\aurora_common.py:42: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Knowledge agent (RAG over PGVector)

In [2]:
retriever = get_knowledge_retriever(k=3)


@tool
def search_help_center(query: str) -> str:
    """Search store help-center policies (returns, refunds, shipping, warranty, membership)."""
    docs = retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs)


knowledge_agent = create_agent(
    model=llm_noreason,
    tools=[search_help_center],
    system_prompt="Answer store-policy questions using ONLY the help-center tool. Cite the policy briefly.",
)

knowledge_agent.invoke(
    {"messages": [HumanMessage(content="How long do refunds take and what's the limit for auto-approval?")]}
)["messages"][-1].pretty_print()

================================== Ai Message ==================================

Refunds are issued to the original payment method within **5-7 business days** after the returned item is received.

Regarding auto-approval, refunds **over $100** require manager approval, implying that refunds of $100 or less are auto-approved.


## Order-data agent (SQL over Postgres)
> **TODO (student):** add a dynamic schema hint (see wk4.1.1) and guard against non-SELECT queries.

In [3]:
orders_db = get_orders_sqldatabase()


@tool
def run_sql(query: str) -> str:
    """Run a read-only SQL query against the orders database. Tables: customers, products, orders, order_items."""
    try:
        return orders_db.run(query)
    except Exception as e:
        return f"Error: {e}"


order_agent = create_agent(
    model=llm_noreason,
    tools=[run_sql],
    system_prompt=(
        "You answer questions about orders and customers using SQL. "
        "Tables: customers(customer_id,name,email,tier), products(product_id,name,price), "
        "orders(order_id,customer_id,status,total,placed_at), order_items(order_id,product_id,quantity). "
        "Only issue SELECT statements."
    ),
)

order_agent.invoke(
    {"messages": [HumanMessage(content="What is the status and total of order 1001, and who placed it?")]}
)["messages"][-1].pretty_print()

================================== Ai Message ==================================

Order 1001 has a status of **shipped** and a total of **$129.49**. It was placed by **Ada Lovelace**.


## Web / research agent
> **TODO (student):** replace `track_shipment` with the real MCP web-search tool from wk4.2
> (`MultiServerMCPClient` over HTTP).

In [4]:
@tool
def track_shipment(order_id: str) -> str:
    """Look up carrier tracking status for an order (simulated)."""
    return f"Order {order_id}: in transit, expected in 2 days (carrier: ACME Express)."


web_agent = create_agent(
    model=llm_noreason,
    tools=[track_shipment],
    system_prompt="You look up shipment and carrier tracking information.",
)

web_agent.invoke(
    {"messages": [HumanMessage(content="Where is my order 1001?")]}
)["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your order 1001 is currently in transit with ACME Express and is expected to arrive in 2 days.


### Definition of done
- Each agent answers a relevant question using its tool.
- Order agent only emits SELECT queries.
- Bonus: web agent backed by the real MCP server.